# 模块 A 综合项目：OpenCV 图像处理与检测交付

用程序生成一张彩色图，再完成中文路径读写、缩放、灰度滤波、颜色目标检测、绘框、统计与 PNG/CSV 交付。

**项目类型：**教师/自学参考答案 · 分步骤完整代码

## 提交要求

1. 生成素材并核对图像尺寸、BGR 通道
2. 通过中文路径读取图片，缩放并做灰度/滤波/Canny
3. HSV 提取红色目标并绘制外接矩形
4. 保存处理图、数据 CSV 与对比 Matplotlib 图片
5. 验证所有文件存在

## 0. 准备工作：执行以下代码生成本地素材

这个单元格可以直接运行。素材为合成测试数据，无需联网。

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

out = Path("A_results")
out.mkdir(exist_ok=True)
img = np.zeros((120,160,3), dtype=np.uint8)
cv2.rectangle(img,(25,20),(90,95),(0,0,255),-1)
cv2.circle(img,(125,60),18,(255,0,0),-1)
ok, raw=cv2.imencode(".png",img)
assert ok
raw.tofile(str(out/"A_原始图片.png"))
print("原图:",img.shape)

## TODO 1：通过中文路径读取图片，缩放并做灰度/滤波/Canny

运行参考实现，观察变量变化与输出。 

In [ ]:
source=out/"A_原始图片.png"
read=cv2.imdecode(np.fromfile(str(source),dtype=np.uint8),cv2.IMREAD_COLOR)
assert read is not None
small=cv2.resize(read,(80,60))
gray=cv2.cvtColor(small,cv2.COLOR_BGR2GRAY)
blur=cv2.GaussianBlur(gray,(5,5),0)
edges=cv2.Canny(blur,40,100)
print("缩放:",small.shape,"边缘数:",np.count_nonzero(edges))

## TODO 2：HSV 提取红色目标并绘制外接矩形

运行参考实现，观察变量变化与输出。 

In [ ]:
hsv=cv2.cvtColor(small,cv2.COLOR_BGR2HSV)
mask=cv2.inRange(hsv,np.array([0,100,100],np.uint8),np.array([10,255,255],np.uint8))
contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
boxes=[cv2.boundingRect(c) for c in contours if cv2.contourArea(c)>20]
visual=small.copy()
for x,y,w,h in boxes:
    cv2.rectangle(visual,(x,y),(x+w,y+h),(0,255,0),2)
print("检测框:",boxes)

## TODO 3：保存处理图、数据 CSV 与对比 Matplotlib 图片

运行参考实现，观察变量变化与输出。 

In [ ]:
assert cv2.imencode(".png",visual)[0]
cv2.imencode(".png",visual)[1].tofile(str(out/"A_目标检测.png"))
pd.DataFrame([{"x":x,"y":y,"w":w,"h":h} for x,y,w,h in boxes]).to_csv(
    out/"A_boxes.csv",index=False,encoding="utf-8-sig"
)
fig,axes=plt.subplots(1,3,figsize=(10,4))
axes[0].imshow(cv2.cvtColor(small,cv2.COLOR_BGR2RGB));axes[0].set_title("Original")
axes[1].imshow(edges,cmap="gray");axes[1].set_title("Canny")
axes[2].imshow(cv2.cvtColor(visual,cv2.COLOR_BGR2RGB));axes[2].set_title("Detection")
for a in axes:a.axis("off")
fig.tight_layout();fig.savefig(out/"A_对比图.png",dpi=150);plt.close(fig)

## TODO 4：验证所有文件存在

运行参考实现，观察变量变化与输出。 

In [ ]:
required=["A_原始图片.png","A_目标检测.png","A_boxes.csv","A_对比图.png"]
for name in required:
    assert (out/name).is_file() and (out/name).stat().st_size>0,name
assert len(boxes)>=1
print("A 模块全部交付文件核验通过")

## 交付核验

- `A_results/A_原始图片.png`
- `A_results/A_目标检测.png`
- `A_results/A_boxes.csv`
- `A_results/A_对比图.png`

完成后核对所有文件是否存在并可打开。参考答案在 `solutions/A_project_solution.ipynb`。

In [ ]:
# 交付文件自检：仅检查文件是否存在且非空，不自动判断模型效果
from pathlib import Path
folder=Path("A_results")
required=["A_原始图片.png","A_目标检测.png","A_boxes.csv","A_对比图.png"]
missing=[name for name in required if not (folder/name).is_file() or (folder/name).stat().st_size==0]
if missing:
    print("缺失或为空的文件：",missing)
else:
    print("全部交付文件已生成")
